# Francis: purchasing and inventory planning

[Open in Colab](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/abw/notebooks/optimization/francis-material-planning.ipynb) · [Open in Binder](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/abw/notebooks/optimization/francis-material-planning.ipynb)

By Joaquim Gromicho. Modernized from the original teaching notebook.

Student case notebook: read the complete case, compute material requirements, and develop the purchasing model. The full worked optimization solution is kept in the private instructor collection.



## Francis' material planning



As we know, Caroline produces football and golf trophies using two types of ornamental balls, plaques, and wood. 

Each trophy has the following consumption of materials:

| Trophy | Wood in meter | Football | Golf ball | Plaque |
|:-------|--------------:|---------:|----------:|-------:|
|Football|           0.4 |        1 |           |      1 |
|Golf    |           0.2 |          |         1 |      1 |

Caroline conducted a data analysis which lead to the following prediction of monthly demands for her trophies: 

| Trophy | Jan | Feb | Mar | Apr | May | Jun | Jul | Aug | Sep | Oct | Nov | Dec |
|:-------|----:|----:|----:|----:|----:|----:|----:|----:|----:|----:|----:|----:|
|Football|  88 | 125 | 260 | 217 | 238 | 286 | 248 | 238 | 265 | 293 | 259 | 244 |
|Golf    |  47 |  62 |  81 |  65 |  95 | 118 |  86 |  89 |  82 |  82 |  84 | 66  |

Caroline hired Francis to plan the materials. 
Francis may buy from each of three suppliers and may also stock materials. 

The suppliers can deliver the following materials:
 - A: **footballs**, **golf balls** and **plaques**
 - B: **wood**
 - C: all of the above

Wood should be acquired in multiples of 10 meter, since it is delivered in poles of 10 meter. 
Ornaments such as balls and plaques may be acquired in any number, but the price is in batches of 100. Meaning that 30 footballs with 10 golf balls and 50 plaques costs as much as 1 football but half of 30 footballs with 30 golf balls and 50 plaques. 
Furthermore, supplier C sells all materials and offers a discount if purchased together: 10 meter of wood and a batch of ornaments cost just 7. This set price is only applied to pairs, meaning that 10 meter of wood and 2 batches cost 13.

The prices are as follows in &euro;:

|Supplier|Wood per 10 meter pole|Batch of ornaments|Together|
|:-------|---------------------:|-----------------:|-------:|
| A      |                    - |                5 |      - |
| B      |                    3 |                - |      - |
| C      |                    4 |                6 |      7 |

When stocking materials, the inventory costs are as follows per month:

|Wood per meter|Football per unit|Golf per unit|Plaque per unit|
|---:|-------:|---:|-----:|
| 0.1|   0.02 |0.02| 0.02 |

The holding price of wood is per meter and the wood stocked is rounded up to full meters, meaning that 12 decimeters pay for 2 meters. 

The capacity limitations of the warehouse allow for a maximum of 1000 meter of wood in stock at any moment.
There are no practical limitations to the number of ornaments in stock.

As you recall, Caroline has the following stock at the moment:

|Wood|Football|Golf|Plaque|
|---:|-------:|---:|-----:|
| 480|   1000 |1500| 1750 |

Caroline would like to have at least the following stock at the end of the year:

|Wood|Football|Golf|Plaque|
|---:|-------:|---:|-----:|
| 200|    500 | 500| 1000 |

Please help Francis to model the material planning for Caroline and solve it with the data above. 

Note that Francis aims at minimizing the acquisition and holding costs of the materials while meeting the required quantities for production. 
The production is made to order, meaning that no inventory of trophies is kept.



## Compute material requirements first
No optimization package is needed for this step. Wood is measured in decimetres in the integer requirement table: 4 and 2 decimetres are 0.4 and 0.2 metres.


In [ ]:
# Use installed packages, install only missing ones, without version pins.
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'pandas': 'pandas', 'numpy': 'numpy'}
ensure_packages(required_packages)


In [ ]:
import pandas as pd
import numpy as np
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
trophy_demand = pd.DataFrame([[88,125,260,217,238,286,248,238,265,293,259,244], [47,62,81,65,95,118,86,89,82,82,84,66]], index=['Football','Golf'], columns=months)
trophy_demand.index.name = 'Trophy'
use = pd.DataFrame({'Football': {'football':1,'golf ball':0,'plaque':1,'wood':4}, 'Golf': {'football':0,'golf ball':1,'plaque':1,'wood':2}})
requirements = use.dot(trophy_demand)
display(trophy_demand, use, requirements)
assert requirements.loc['wood','Jan'] == 446  # 44.6 metres
assert requirements.loc['plaque','Jan'] == 135


## Build the model in stages

1. Choose units for wood and ornaments. Explain why demand for materials follows from the matrix multiplication above.
2. Define purchases by supplier and month, and end-of-month inventory. Identify integer quantities: poles, paid ornament batches and paired discounts.
3. Write stock balances from initial inventory through December, enforcing the final-stock requirements.
4. Model the paired discount at supplier C. Distinguish quantities delivered from paid batches, and avoid multiplying decision variables.
5. State when deliveries arrive and when production and holding costs occur. Enforce the 1000-metre warehouse limit at the time stock is highest.
6. Minimize acquisition and holding cost, including whole-metre wood holding charges.

The case leaves timing conventions implicit. State a consistent convention before solving; do not silently treat a final inventory level as the maximum stock during a month.


## Introduce Pyomo when you start modeling
The data calculation did not require an optimization package. The next cell prepares one now. Add your variables, objective and constraints below; this notebook does not contain the instructor’s full-case solution.


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'pyomo': 'pyomo', 'highspy': 'highspy'}
ensure_packages(required_packages)


In [ ]:
import pyomo.environ as pyo
from teaching_utils import available_pyomo_solvers
print(available_pyomo_solvers(['appsi_highs']))
model=pyo.ConcreteModel('Francis: student purchasing model')
model.months=pyo.Set(initialize=months,ordered=True)
model.materials=pyo.Set(initialize=list(requirements.index))
# Add your variables and stock-balance rules here.


## Check your proposed plan

Check every monthly material balance, purchase-domain condition, capacity bound and final-stock requirement. Recompute acquisition and holding costs independently from the returned plan. Compare a policy with no advance purchasing to an optimized inventory policy, using the same timing convention in both. Report solver termination before interpreting the objective.

Extension: compare a formulation with explicit paired purchases against one using a separate discount-count variable. Explain why their feasible purchasing plans and costs should agree.
